[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb2_train_video_only.ipynb)

# nb2: train the video-only model
Trains `video_only_s0` (spec section 2). Safe to rerun: a finished run (`log.json` present) is skipped, an interrupted run resumes from its newest `resume_*.pt`, and nothing on Drive is ever overwritten. 

**Run `nb0_benchmark` first**: it builds the data cache and decides FVD vs KID once for every run.

## 1. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> A100 GPU, then rerun.")
print("cuda:", torch.cuda.get_device_name(0), "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 3. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 4. Install

In [ ]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

## 5. Check nb0 outputs exist

In [ ]:
import os
for f in ["cache/pendulum_T64_dt0.04_train.npz", "cache/pendulum_T64_dt0.04_heldout.npz", "cache/video_metric.json"]:
    assert os.path.exists(f"{ROOT}/{f}"), f"missing {f}: run nb0_benchmark first"
print("nb0 outputs present")

## 6. Train `video_only_s0`
Prints the 200-step ETA, then one line per checkpoint (every 2k steps). Console output is also saved to a timestamped log on Drive.

In [ ]:
%%bash
set -o pipefail
mkdir -p "/content/drive/MyDrive/physgen_cvpr27/video_only_s0"
cd /content/repo
PYTHONUNBUFFERED=1 python -u -m cvpr27.train --variant video_only --run_name video_only_s0 --seed 0 2>&1 | tee -a "/content/drive/MyDrive/physgen_cvpr27/video_only_s0/console_$(date +%s).log"

## 7. Summary (read from the run's files)

In [ ]:
!cd /content/repo && python -m cvpr27.summary --run video_only_s0